# Notebook Summary

- This notebook converts exported timing data into analysis-ready wide tables.
- It reshapes repeated measurements by trial, mapping, and layer configuration.
- The generated tables can be used directly for statistical modeling and reporting.


## Setup and Imports

- This block imports required Python libraries for data processing and visualization.
- It defines global paths and constants used consistently across subsequent cells.


In [1]:
# Imports / global constants

# CSV files are located in ../data

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import math
import numpy as np
import seaborn as sns

from pandas.plotting import parallel_coordinates

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"

path = "../data/"
all_files = glob.glob(path + "/*.csv")


## Load and Inspect Data

- This block loads one or more CSV/TSV sources into pandas dataframes.
- It performs a quick inspection to validate schema and content before transformations.


In [2]:
duration = pd.read_csv(rf'{export}times.csv', sep= ";")

display(duration)


,Unnamed: 0,MappingMethod,FrameNumber_Start,FrameNumber_Finish,Block,Task,Trial,NumLayers,Target,Target_Relative,DateTime_Start,DateTime_Finish,Duration,NumFrames,Result,Proband,FrameRate_1
0,0,direct,65,242,1,1,0,12,3,0.250000,2021-06-29T07:19:27.465Z,2021-06-29T07:19:34.185Z,6.720,177,COMPLETED,23,26.339286
1,1,direct,295,547,1,1,1,12,11,0.916667,2021-06-29T07:19:36.096Z,2021-06-29T07:19:45.693Z,9.597,252,COMPLETED,23,26.258206
2,2,direct,604,808,1,1,2,12,9,0.750000,2021-06-29T07:19:47.749Z,2021-06-29T07:19:55.418Z,7.669,204,COMPLETED,23,26.600600
3,3,direct,849,1050,1,1,3,12,6,0.500000,2021-06-29T07:19:56.961Z,2021-06-29T07:20:04.496Z,7.535,201,TERMINATED,23,26.675514
4,4,direct,1114,1350,1,1,4,12,1,0.083333,2021-06-29T07:20:06.928Z,2021-06-29T07:20:16.018Z,9.090,236,COMPLETED,23,25.962596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,6205,widening,88872,89061,3,18,0,15,8,0.533333,2021-05-31T10:18:07.589Z,2021-05-31T10:18:14.593Z,7.004,189,COMPLETED,12,26.984580
6206,6206,widening,89096,89278,3,18,1,15,11,0.733333,2021-05-31T10:18:15.806Z,2021-05-31T10:18:22.355Z,6.549,182,COMPLETED,12,27.790502
6207,6207,widening,89320,89518,3,18,2,15,14,0.933333,2021-05-31T10:18:23.773Z,2021-05-31T10:18:31.158Z,7.385,198,TERMINATED,12,26.811104
6208,6208,widening,89570,89735,3,18,3,15,4,0.266667,2021-05-31T10:18:32.948Z,2021-05-31T10:18:38.840Z,5.892,165,TERMINATED,12,28.004073


## Reshape to Wide Format

- This block pivots repeated measurements into a wide format matrix.
- The resulting structure supports condition-level comparison and statistical testing.


In [33]:
# Reshape long-format measurements into a wide table for condition-wise comparison.
duration_mapping = duration.pivot_table(columns=['Trial','MappingMethod', 'NumLayers', ], index=['Proband'], values=['Duration'])

# Combine intermediate tables to align metrics on shared identifiers.
duration_mapping.columns = ['_'.join(map(str, col)) for col in duration_mapping.columns]
display(duration_mapping)
display(duration_mapping.columns)

duration_mapping.to_csv(rf'{export}duration_repeated_columns.csv', sep= ";")


,Duration_0_ densening_6,Duration_0_ densening_9,Duration_0_ densening_12,Duration_0_ densening_15,Duration_0_ densening_18,Duration_0_ densening_21,Duration_0_ direct_6,Duration_0_ direct_9,Duration_0_ direct_12,Duration_0_ direct_15,...,Duration_4_ direct_12,Duration_4_ direct_15,Duration_4_ direct_18,Duration_4_ direct_21,Duration_4_ widening_6,Duration_4_ widening_9,Duration_4_ widening_12,Duration_4_ widening_15,Duration_4_ widening_18,Duration_4_ widening_21
Proband,,,,,,,,,,,,,,,,,,,,,
1,6.442667,6.523000,7.272333,9.031333,6.681000,6.330000,6.295667,7.859667,6.161667,8.141000,...,6.657333,6.968333,8.542000,24.817000,4.913333,6.674667,7.033667,10.355333,7.527000,7.349000
2,6.301667,8.634667,8.458000,8.942000,23.410333,7.199333,6.220333,8.098667,9.784000,7.107333,...,7.861333,9.329333,12.740667,10.708000,8.698333,8.332667,7.976000,15.760333,21.301333,12.288667
3,7.168333,7.495667,8.517333,11.577667,12.058000,16.034667,13.159333,6.477667,8.211667,6.335667,...,11.134000,10.029000,14.718000,8.006000,5.381667,5.277000,7.648667,7.862000,9.703667,12.514333
4,7.670667,12.678000,8.272000,10.196333,23.330000,18.143333,5.693000,8.280333,8.941333,10.879000,...,8.400667,14.667333,25.300667,16.796000,5.423333,14.393333,16.759000,13.405333,18.512667,12.185667
5,5.018000,6.165667,9.013667,8.446667,8.555333,7.510333,4.382000,5.291667,7.351000,10.867333,...,6.391667,7.321000,6.865667,7.035000,7.470333,4.986667,7.809667,6.857333,10.222667,8.460000
6,6.293667,5.742667,11.266000,8.882667,7.607333,9.545000,6.178333,5.817333,6.941333,8.736333,...,6.642000,8.089667,9.615333,10.981333,6.450667,13.166667,9.378000,8.564333,8.136000,13.460333
7,6.803000,6.784667,6.940333,8.063667,8.173000,9.506333,6.134667,19.091333,6.889667,12.581667,...,8.578000,9.291000,8.316667,13.932333,5.877333,9.683333,8.015667,10.828333,8.548000,7.461000
8,13.982000,18.782667,14.282000,12.278667,12.862667,13.315333,7.290333,11.106000,7.828667,24.806667,...,6.743667,11.468333,9.100333,9.106667,4.863000,8.051000,8.396333,9.865667,15.138667,19.217333
9,7.010333,14.763333,8.057667,12.870000,7.161667,11.830000,8.821667,5.665333,8.723000,8.512333,...,8.009333,12.456667,10.810667,8.380333,6.340333,6.672667,7.388000,9.360667,8.186667,11.987333


Index(['Duration_0_ densening_6', 'Duration_0_ densening_9',
       'Duration_0_ densening_12', 'Duration_0_ densening_15',
       'Duration_0_ densening_18', 'Duration_0_ densening_21',
       'Duration_0_ direct_6', 'Duration_0_ direct_9', 'Duration_0_ direct_12',
       'Duration_0_ direct_15', 'Duration_0_ direct_18',
       'Duration_0_ direct_21', 'Duration_0_ widening_6',
       'Duration_0_ widening_9', 'Duration_0_ widening_12',
       'Duration_0_ widening_15', 'Duration_0_ widening_18',
       'Duration_0_ widening_21', 'Duration_1_ densening_6',
       'Duration_1_ densening_9', 'Duration_1_ densening_12',
       'Duration_1_ densening_15', 'Duration_1_ densening_18',
       'Duration_1_ densening_21', 'Duration_1_ direct_6',
       'Duration_1_ direct_9', 'Duration_1_ direct_12',
       'Duration_1_ direct_15', 'Duration_1_ direct_18',
       'Duration_1_ direct_21', 'Duration_1_ widening_6',
       'Duration_1_ widening_9', 'Duration_1_ widening_12',
       'Duration_1_ w

## Reshape to Wide Format

- This block pivots repeated measurements into a wide format matrix.
- The resulting structure supports condition-level comparison and statistical testing.


In [32]:
# Reshape long-format measurements into a wide table for condition-wise comparison.
duration_mapping2 = duration.pivot_table(columns=['MappingMethod', 'NumLayers'], index=['Proband', 'Trial'], values=['Duration']).reset_index()

# Combine intermediate tables to align metrics on shared identifiers.
duration_mapping2.columns = ['_'.join(map(str, col)) for col in duration_mapping2.columns]
display(duration_mapping2)
display(duration_mapping2.columns)

duration_mapping2.to_csv(rf'{export}duration_repeated_columns_trial.csv', sep= ";")


,Proband__,Trial__,Duration_ densening_6,Duration_ densening_9,Duration_ densening_12,Duration_ densening_15,Duration_ densening_18,Duration_ densening_21,Duration_ direct_6,Duration_ direct_9,Duration_ direct_12,Duration_ direct_15,Duration_ direct_18,Duration_ direct_21,Duration_ widening_6,Duration_ widening_9,Duration_ widening_12,Duration_ widening_15,Duration_ widening_18,Duration_ widening_21
0,1,0,6.442667,6.523000,7.272333,9.031333,6.681000,6.330000,6.295667,7.859667,6.161667,8.141000,7.027333,7.777667,5.608333,5.769000,5.619333,6.728667,12.254667,7.306333
1,1,1,6.391333,6.341667,6.181667,6.281333,8.007667,7.966000,5.119333,6.731000,11.839667,8.448000,6.697667,8.550333,5.020333,6.273667,7.872333,8.589667,9.326333,9.176667
2,1,2,4.664000,6.035667,6.654667,7.336000,8.615667,9.626667,6.047667,9.462000,7.018667,8.031333,8.234333,7.516667,3.617667,5.852333,6.204667,9.912000,7.639333,7.865000
3,1,3,5.513000,4.257667,7.054667,6.897000,6.115000,8.149000,4.867667,7.622333,5.719000,7.036667,6.065667,10.393000,5.331667,13.129333,6.121667,6.176667,9.312667,7.945000
4,1,4,4.504000,5.250667,6.707333,7.959667,9.580667,7.045667,5.040000,8.154000,6.657333,6.968333,8.542000,24.817000,4.913333,6.674667,7.033667,10.355333,7.527000,7.349000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,24,0,6.866667,8.004000,7.531333,7.550667,7.207333,11.735667,5.030000,6.139667,8.169667,7.365667,7.736333,6.039333,5.340667,6.166000,7.653667,6.250000,8.783667,7.700333
111,24,1,6.595667,5.969667,10.485667,7.598000,12.053333,8.849000,5.943333,5.446000,6.538333,8.823333,8.634333,7.736333,6.553000,6.199333,8.510333,7.386333,6.247000,10.576000
112,24,2,6.264333,6.929000,9.015000,9.751000,7.576333,10.948667,5.663667,5.492000,8.008000,6.352333,7.208333,9.041000,4.279667,5.818667,7.751000,6.240667,9.013333,7.804333
113,24,3,5.911000,11.263000,6.539667,7.656333,6.632000,15.616333,5.241667,6.855333,8.574667,8.244000,8.464667,9.700667,4.994667,5.465333,6.997000,6.557667,6.074000,5.708000


Index(['Proband__', 'Trial__', 'Duration_ densening_6',
       'Duration_ densening_9', 'Duration_ densening_12',
       'Duration_ densening_15', 'Duration_ densening_18',
       'Duration_ densening_21', 'Duration_ direct_6', 'Duration_ direct_9',
       'Duration_ direct_12', 'Duration_ direct_15', 'Duration_ direct_18',
       'Duration_ direct_21', 'Duration_ widening_6', 'Duration_ widening_9',
       'Duration_ widening_12', 'Duration_ widening_15',
       'Duration_ widening_18', 'Duration_ widening_21'],
      dtype='object')